# 04 - Next Product to Buy
Goal: for each registered customers, recommend 5 products they have not bought before, and measure honestly (time-based split) whether they buy them in the next 3 month. Compare a popularity baseline, an association-rule recommender and an item-similarity recommender using Hit Rate@5, Precision@5 and Recall@5.

## 1. Load data and time split

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 60)

PROJECT_DIR = Path.cwd().parent
df= pd.read_parquet(PROJECT_DIR / "data" / "processed" / "online_retail_clean.parquet")

CUTOFF   = pd.Timestamp("2011-09-01")
TEST_END = pd.Timestamp("2011-12-01")
K = 5

reg   = df[df["Customer ID"].notna()]
hist  = reg[reg["InvoiceDate"] < CUTOFF]
test  = reg[(reg["InvoiceDate"] >= CUTOFF) & (reg["InvoiceDate"] < TEST_END)]

print("History:", hist["InvoiceDate"].min().date(), "->", hist["InvoiceDate"].max().date(),
      "|", f"{len(hist):,} lines")
print("Test:   ", test["InvoiceDate"].min().date(), "->", test["InvoiceDate"].max().date(),
      "|", f"{len(test):,} lines")

History: 2009-12-01 -> 2011-08-31 | 603,313 lines
Test:    2011-09-01 -> 2011-11-30 | 150,099 lines


I pretend that today is September 2011. Everything before that date is history, and the models are only allowed to learn from it. The next three months (September to November) are the test period, where I check what customers actually bought. December 2011 is left out because the data stops on the 9th. Splitting by the time, instead of randomly, is what makes the test honet: in real life a model is always buikt on the past and used on the future.

In [2]:
hist_items = hist.groupby("Customer ID")["StockCode"].apply(set)   # products bought before
test_items = test.groupby("Customer ID")["StockCode"].apply(set)   # products bought in test

eval_ids = hist_items.index.intersection(test_items.index)         # active in both periods

new_items = {c: test_items[c] - hist_items[c] for c in eval_ids}   # first-time purchases only
new_items = {c: s for c, s in new_items.items() if len(s) > 0}     # keep customers with >=1 new product

n_new = pd.Series({c: len(s) for c, s in new_items.items()})
print("Customers with history:             ", f"{len(hist_items):,}")
print("Active in both periods:             ", f"{len(eval_ids):,}")
print("...of whom bought >= 1 NEW product: ", f"{len(new_items):,}")
print("\nNew products per evaluated customer:")
print(n_new.describe().round(1))

Customers with history:              5,216
Active in both periods:              2,241
...of whom bought >= 1 NEW product:  2,150

New products per evaluated customer:
count    2150.0
mean       29.4
std        38.0
min         1.0
25%        10.0
50%        20.0
75%        38.0
max      1072.0
dtype: float64


2,150 customers bought something both before and after the cutoff, and bought at leastone product they had never bought before. These are the customers I evaluate. I only count *new* products as correct answer. Wholesale customers re-order the same items all the time, so a model that simply says "buy what you bought last time" would look very good without doing any real cross-selling. A typical customer bought 20 new products in the test period, out of roughly 4,600 they had never tried.

In [3]:
def evaluate(recommend, name):
    """Run a recommender for every evaluated custmer and average the metrics."""
    hits, precision, recall = [], [], []
    for c, truth in new_items.items():          # truth = new products the customer really bought
        recs = recommend(c)                     # our K recommendations
        n_hit = len(set(recs) & truth)          # how many of them were bought
        hits.append(n_hit > 0)
        precision.append(n_hit / K)
        recall.append(n_hit / len(truth))
    return pd.Series({"model": name,
                      "HitRate@5": np.mean(hits),
                      "Precision@5": np.mean(precision),
                      "Recall@5": np.mean(recall),
                      "customers": len(hits)})

In [4]:
names = df.drop_duplicates("StockCode").set_index("StockCode")["Description"]
rng = np.random.default_rng(42)
all_items = hist["StockCode"].unique()

def make_popular(ranking):
    """Build a recommender that gives each customer the top-ranked products they don't own yet."""
    ranked = list(ranking.index)
    def recommend(c):
        owned = hist_items[c]
        out = []
        for item in ranked:
            if item not in owned:
                out.append(item)
                if len(out) == K:
                    break
        return out
    return recommend

def recommend_random(c):
    owned = hist_items[c]
    picks =[i for i in rng.choice(all_items, 50, replace=False) if i not in owned]
    return picks[:K]

# popularity = number of different customers who bought the product
pop_all = hist.groupby("StockCode")["Customer ID"].nunique().sort_values(ascending=False)

recent = hist[hist["InvoiceDate"] >= CUTOFF - pd.DateOffset(months=3)]    # Jun-Aug 2011
pop_recent = recent.groupby("StockCode")["Customer ID"].nunique().sort_values(ascending=False)

print("Top 10 recent best-sellers:")
print(names[pop_recent.index[:10]].to_string())

Top 10 recent best-sellers:
StockCode
47566                          PARTY BUNTING
23298                         SPOTTY BUNTING
85123A    WHITE HANGING HEART T-LIGHT HOLDER
23245             SET OF 3 REGENCY CAKE TINS
22423               REGENCY CAKESTAND 3 TIER
23203               JUMBO BAG VINTAGE DOILY 
85099B               JUMBO BAG RED RETROSPOT
23209               LUNCH BAG VINTAGE DOILY 
20725                LUNCH BAG RED RETROSPOT
22384                LUNCH BAG PINK POLKADOT


Picking 5 products at random gives a hit rate of 2.6%, close  to the 2% I expected from simple probability, which tells me the evaluation code works. Recommending the best-sellers is about eight times better: roughly one customer in five buys at least one of them. I expected the "last 3 months" list to do worse, because it is full of summer items going into the Chirstmas season, but it actually beat the all-time list (20.8% vs 18.6%). What is popular right now is a better guide than a two-year average. This 20.8% is the number any real model has to beat.

Recal stays around 1% fro every model, simply because customers buy about 20 new products and we only suggest 5. SO hit rate and precision are the more useful measuers here.

In [5]:
%%time
results = pd.DataFrame([
    evaluate(recommend_random,         "Random"),
    evaluate(make_popular(pop_all),     "Popularity (all history)"),
    evaluate(make_popular(pop_recent), "Popularity (last 3 months)"),
]).set_index("model")
results.round(3)

CPU times: total: 844 ms
Wall time: 840 ms


,HitRate@5,Precision@5,Recall@5,customers
model,,,,
Random,0.026,0.005,0.001,2150
Popularity (all history),0.186,0.045,0.010,2150
Popularity (last 3 months),0.208,0.055,0.011,2150


In [6]:
truth_all = set().union(*new_items.values())
unseen = {i for i in truth_all if i not in set(all_items)}
share = np.mean([len(s & unseen) / len(s) for s in new_items.values()])
print("Products bought in test that never appeared in history:", len(unseen))
print(f"Average share of a customer's new purchases that are brand-new products: {share:.1%}")

Products bought in test that never appeared in history: 214
Average share of a customer's new purchases that are brand-new products: 15.0%


214 products were launched after the cutoff, and on average 15% of a customer's new purchases are products that did not exist in the history. No model built on past data can recommend something it has never seen, so part of the "truth" is out of reach for every approach. This is a natural limit of the test, not a flaw in any one model.

In [7]:
from mlxtend.frequent_patterns import fpgrowth, association_rules

MIN_SUPPORT, MAX_ITEMS = 0.01, 200

hb = df[df["InvoiceDate"] < CUTOFF]              # all history baskets (guests too)
size_h = hb.groupby("Invoice")["StockCode"].nunique()
hb = hb[hb["Invoice"].isin(size_h[size_h <= MAX_ITEMS].index)]    # cap mega-baskets, as in 03

n_b = hb["Invoice"].nunique()
sup = hb.groupby("StockCode")["Invoice"].nunique() / n_b
freq = sup[sup >= MIN_SUPPORT].index

basket_h = (hb[hb["StockCode"].isin(freq)]
            .drop_duplicates(["Invoice", "StockCode"])
            .assign(bought = True)
            .pivot(index = "Invoice", columns = "StockCode", values = "bought")
            .notna()
            .reindex(hb["Invoice"].unique(), fill_value = False))

itemsets_h = fpgrowth(basket_h, min_support = MIN_SUPPORT, use_colnames = True)
rules_h = association_rules(itemsets_h, num_itemsets = len(basket_h),
                           metric = "lift", min_threshold = 1)

rule_list = list(zip(rules_h["antecedents"], rules_h["consequents"], rules_h["confidence"]))
print("History baskets:", f"{n_b:,}", "| frequent products:", len(freq), "| rules:", len(rules_h))

History baskets: 31,636 | frequent products: 621 | rules: 1290


The association rules used here are rebuilt from scratch using only baskets from before the cutoff, with the same method as notebook 03 (mega-baskets removed, products with at least 1% support, FP-Growth). I did not reuse the rules saved in notebook 03, because those were learned from the full dataset including September to November, which would let the model peek at the answers. This gives 1,290 rules over 621 products.

In [8]:
fallback = list(pop_recent.index)                # best baseline, used to fill empty slots

def recommend_rules(c):
    owned = hist_items[c]
    scores = {}
    for ante, cons, conf in rule_list:
        if ante <= owned:                         # customer already bought ALL "if" products
            for item in cons:
                if item not in owned:             # recommend only NEW products
                    scores[item] = max(scores.get(item, 0), conf)
    recs = sorted(scores, key=scores.get, reverse=True)[:K]
    for item in fallback:                         # fill up to K with best-sellers
        if len(recs) == K:
            break
        if item not in owned and item not in recs:
            recs.append(item)
    return recs

# how often do rules along give at least one / five recommendations?
n_from_rules = []
for c in new_items:
    owned = hist_items[c]
    cand = {i for a, cons, _ in rule_list if a <= owned for i in cons if i not in owned}
    n_from_rules.append(len(cand))
n_from_rules = pd.Series(n_from_rules)
print(f"Customers with >= 1 rule-based candidate: {(n_from_rules >= 1).mean():.1%}")
print(f"Customers with >= 5 rule-based candidates: {(n_from_rules >= K).mean():.1%}")

Customers with >= 1 rule-based candidate: 96.4%
Customers with >= 5 rule-based candidates: 88.7%


For each customer, the recommender looks at every rule whose "if" products they already own, collects the "then" products they have not bought yet, and ranks them by the rule's confidence. If there are fewer than five, the list is topped up with recent best-sellers. The rules turn out to cover almost everyone: 96% of customers get at least one rule-based suggestion and 89% get all five, so the results below mostly reflect the rules themselves rather than the fallback.

In [9]:
results.loc["Association rules (+ popularity fill)"] = evaluate(recommend_rules, "Association rules").drop("model")
results.round(3)

,HitRate@5,Precision@5,Recall@5,customers
model,,,,
Random,0.026,0.005,0.001,2150
Popularity (all history),0.186,0.045,0.010,2150
Popularity (last 3 months),0.208,0.055,0.011,2150
Association rules (+ popularity fill),0.255,0.067,0.017,2150


The rule-based recommender beats the best baseline on every measure: a hit rate of 25.5% against 20.8%, which means about 23% more customers bought at least one of the suggested new products. Precision rises from 5.5% to 6.7%, and recal from 5.5% to 6.7%, and recal from 1.1% to 1.7%. The rules are picking up personal patterns (what goes with what this customer already buys) that a single best-seller list cannot. The gain is real but modest, which is what I would expect: popular products are popular for a reason, and they remain a strong default.